In [1]:
!pip install dagshub mlflow optuna -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 70.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

dagshub.init (
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("Neural Networks")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d7ba8d0e-87ea-4704-bcf1-15f3416e2e70&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=32eed9fa9a31244ac066be059ed2b0022b74ea2317960fd95d0605da91ec4f1e




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


# Cleaning

In [3]:
with mlflow.start_run(run_name="NeuralNetworks_Cleaning"):
    import gc
    import numpy as np
    import pandas as pd

    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD = 0.80
    ID_MISSING_THRESHOLD = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    # load
    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    # fix id-01 vs id_01
    test_id.columns = test_id.columns.str.replace("-", "_", regex=False)

    # merge
    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id, on="TransactionID", how="left")

    del train_txn, train_id, test_txn, test_id
    gc.collect()

    # split target
    y_train = train["isFraud"].astype(np.int8)
    X_train = train.drop(columns=["isFraud", "TransactionID"])
    X_test  = test.drop(columns=["TransactionID"])

    del train, test
    gc.collect()

    # drop high-missing
    id_like_cols = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train.columns if c not in id_like_cols]
    missing_ratio = X_train.isnull().mean()

    drop_txn = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id  = [c for c in id_like_cols if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing = drop_txn + drop_id

    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    # drop near-constant
    near_constant_cols = []
    for col in X_train.columns:
        top_freq = X_train[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train = X_train.drop(columns=near_constant_cols)
    X_test  = X_test.drop(columns=[c for c in near_constant_cols if c in X_test.columns])

    # align test columns
    for col in X_train.columns:
        if col not in X_test.columns:
            X_test[col] = np.nan
    X_test = X_test[X_train.columns]

    # log
    mlflow.log_param("txn_missing_threshold", TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold", ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)

    mlflow.log_metric("train_rows", int(X_train.shape[0]))
    mlflow.log_metric("test_rows", int(X_test.shape[0]))
    mlflow.log_metric("final_features", int(X_train.shape[1]))
    mlflow.log_metric("fraud_rate", float(y_train.mean()))
    mlflow.log_metric("dropped_missing_cols", int(len(drop_missing)))
    mlflow.log_metric("dropped_near_constant_cols", int(len(near_constant_cols)))

    print(f"X_train_clean: {X_train.shape}")
    print(f"X_test_clean:  {X_test.shape}")

    # keep for next cells
    X_train_clean = X_train
    X_test_clean = X_test
    y_train_clean = y_train

X_train_clean: (590540, 353)
X_test_clean:  (506691, 353)
🏃 View run NeuralNetworks_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/4ba1a62f6674400392bcb4694986fca4
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


# Feature Engineering

In [4]:
with mlflow.start_run(run_name="NeuralNetworks_FeatureEngineering"):
    from sklearn.impute import SimpleImputer

    X_train = X_train_clean.copy()
    X_test = X_test_clean.copy()
    y_train = y_train_clean.copy()

    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"] = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"] = np.sin(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_cos"] = np.cos(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test = X_test.drop(columns=["TransactionDT"], errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    num_imp = SimpleImputer(strategy="median")
    X_train[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test[num_cols] = num_imp.transform(X_test[num_cols])

    cat_maps = {}
    for c in cat_cols:
        uniq = pd.Series(X_train[c].astype(str).unique())
        mapping = {v: i for i, v in enumerate(uniq)}
        cat_maps[c] = mapping
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c] = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_metric("features_after_fe", int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded", int(len(cat_cols)))

    print("X_train_fe:", X_train.shape)
    print("X_test_fe: ", X_test.shape)

    X_train_fe_nn = X_train
    X_test_fe_nn = X_test
    y_train_fe_nn = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run NeuralNetworks_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/aeeac25e4d07418bb400cac4b1d97bd4
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


In [5]:
print(X_train_fe_nn.shape, X_test_fe_nn.shape)
assert X_train_fe_nn.shape[1] == X_test_fe_nn.shape[1]
print("object cols left:", X_train_fe_nn.select_dtypes(include=["object"]).shape[1])

(590540, 355) (506691, 355)
object cols left: 0


# Feature Selection

In [6]:
with mlflow.start_run(run_name="NeuralNetworks_FeatureSelection"):
    X_train = X_train_fe_nn.copy()
    X_test = X_test_fe_nn.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test = X_test.drop(columns=const_cols, errors="ignore")

    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    sample_n = min(120_000, len(X_train))
    idx = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()

    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > 0.98).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test = X_test.drop(columns=drop_corr, errors="ignore")

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_metric("dropped_const", len(const_cols))
    mlflow.log_metric("dropped_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print("X_train_fs:", X_train.shape)

    X_train_final_nn = X_train
    X_test_final_nn = X_test

X_train_fs: (590540, 306)
🏃 View run NeuralNetworks_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/9d5413ecd0d9403cb47b3558b925878c
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


In [7]:
print(X_train_final_nn.shape, X_test_final_nn.shape)
assert X_train_final_nn.shape[1] == X_test_final_nn.shape[1]

(590540, 306) (506691, 306)


# **Training**

In [8]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

SAMPLE_SIZE = 150_000
sample_idx = np.random.RandomState(42).choice(len(X_train_final_nn), size=SAMPLE_SIZE, replace=False)
X_sample = X_train_final_nn.iloc[sample_idx].reset_index(drop=True)
y_sample = y_train_fe_nn.iloc[sample_idx].reset_index(drop=True)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_sample, y_sample,
    test_size=0.2, random_state=42, stratify=y_sample
)
print(f"Train: {X_tr.shape} | Val: {X_val.shape}")
print(f"Fraud rate: {y_sample.mean():.4f}")

Train: (120000, 306) | Val: (30000, 306)
Fraud rate: 0.0359


In [9]:
with mlflow.start_run(run_name="MLP_Baseline"):
    mlflow.log_param("hidden_layer_sizes", "(100,)")
    mlflow.log_param("activation",         "relu")
    mlflow.log_param("solver",             "adam")
    mlflow.log_param("max_iter",           100)
    mlflow.log_param("sample_size",        SAMPLE_SIZE)
    mlflow.log_param("note",               "single hidden layer, default params")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    MLPClassifier(hidden_layer_sizes=(100,), activation="relu",
                                  solver="adam", max_iter=100, random_state=42))
    ])
    pipe.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(train_auc - val_auc, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")

[Baseline] Train: 0.9948 | Val: 0.8217 | Gap: 0.1731
🏃 View run MLP_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/9ad904853dee41deac072c7054e580c7
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


In [10]:
architectures = [
    (100,),
    (128, 64),
    (256, 128),
]

arch_results = []
for arch in architectures:
    label = "x".join(str(x) for x in arch)
    with mlflow.start_run(run_name=f"MLP_arch_{label}"):
        mlflow.log_param("hidden_layer_sizes", label)
        mlflow.log_param("activation",         "relu")
        mlflow.log_param("solver",             "adam")
        mlflow.log_param("max_iter",           150)
        mlflow.log_param("sample_size",        SAMPLE_SIZE)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    MLPClassifier(hidden_layer_sizes=arch, activation="relu",
                                      solver="adam", max_iter=150, random_state=42))
        ])
        pipe.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        arch_results.append({"arch": label, "train_auc": train_auc,
                              "val_auc": val_auc, "gap": gap})
        print(f"  {label:<15} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

arch_df = pd.DataFrame(arch_results).set_index("arch")
print(f"\n-> Best arch by val AUC: {arch_df['val_auc'].idxmax()}")
print(arch_df.to_string())

  100             | Train: 0.9960 | Val: 0.8249 | Gap: 0.1710
🏃 View run MLP_arch_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/08a0e905bf614496925517d7a009fab4
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  128x64          | Train: 0.9989 | Val: 0.8264 | Gap: 0.1725
🏃 View run MLP_arch_128x64 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/39d839b91eb54ec294e6d7c20d59c4cc
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  256x128         | Train: 0.9996 | Val: 0.8533 | Gap: 0.1463
🏃 View run MLP_arch_256x128 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/81b3c71f19f94a38b96030907358a074
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5

-> Best arch by val AUC: 256x128
         train_auc   val_auc       gap
ar

In [11]:
best_arch = (256, 128)
best_activation = "relu"
print(f"Using arch: {best_arch}, activation: {best_activation}")

Using arch: (256, 128), activation: relu


In [12]:
alpha_results = []
for alpha in [0.0001, 0.001, 0.01, 0.1]:
    with mlflow.start_run(run_name=f"MLP_alpha_{alpha}"):
        mlflow.log_param("hidden_layer_sizes", str(best_arch))
        mlflow.log_param("activation",         best_activation)
        mlflow.log_param("alpha",              alpha)
        mlflow.log_param("solver",             "adam")
        mlflow.log_param("max_iter",           150)
        mlflow.log_param("sample_size",        SAMPLE_SIZE)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    MLPClassifier(hidden_layer_sizes=best_arch,
                                      activation=best_activation,
                                      alpha=alpha, solver="adam",
                                      max_iter=150, random_state=42))
        ])
        pipe.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        alpha_results.append({"alpha": alpha, "train_auc": train_auc,
                               "val_auc": val_auc, "gap": gap})
        print(f"  alpha={alpha:<6} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

alpha_df = pd.DataFrame(alpha_results).set_index("alpha")
print(f"\n-> Best alpha: {alpha_df['val_auc'].idxmax()}")
print(alpha_df.to_string())

  alpha=0.0001 | Train: 0.9996 | Val: 0.8533 | Gap: 0.1463
🏃 View run MLP_alpha_0.0001 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/3f515b28341741669e7c24ac8999f0ac
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  alpha=0.001  | Train: 0.9994 | Val: 0.8554 | Gap: 0.1441
🏃 View run MLP_alpha_0.001 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/4a8889d9498d4dfca05c0b81ab64af6c
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  alpha=0.01   | Train: 0.9981 | Val: 0.8622 | Gap: 0.1359
🏃 View run MLP_alpha_0.01 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/232123fe166846aab4cacf7150a5ddfd
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
  alpha=0.1    | Train: 0.9578 | Val: 0.8805 | Gap: 0.0773
🏃 View run MLP_alpha_0.

In [13]:
best_alpha = 0.1
print(f"Using alpha: {best_alpha}")


Using alpha: 0.1


In [15]:
with mlflow.start_run(run_name="MLP_CrossValidation_5fold"):
    mlflow.log_param("hidden_layer_sizes", str(best_arch))
    mlflow.log_param("activation",         best_activation)
    mlflow.log_param("alpha",              best_alpha)
    mlflow.log_param("solver",             "adam")
    mlflow.log_param("max_iter",           150)
    mlflow.log_param("cv_folds",           5)
    mlflow.log_param("cv_sample_size",     SAMPLE_SIZE)

    pipe_cv = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    MLPClassifier(hidden_layer_sizes=best_arch,
                                  activation=best_activation,
                                  alpha=best_alpha, solver="adam",
                                  max_iter=150, random_state=42))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe_cv, X_sample, y_sample,
                                cv=cv, scoring="roc_auc")

    for i, score in enumerate(cv_scores):
        mlflow.log_metric("fold_auc", round(score, 5), step=i)

    mlflow.log_metric("cv_mean_auc", round(cv_scores.mean(), 5))
    mlflow.log_metric("cv_std_auc",  round(cv_scores.std(),  5))

    print(f"CV folds: {[round(s, 4) for s in cv_scores]}")
    print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")

🏃 View run MLP_CrossValidation_5fold at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/0ea3bc6ed06d47b190039111834dec3e
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


KeyboardInterrupt: 

In [16]:
best_arch = (256, 128)
best_activation = "relu"
best_alpha = 0.1
print(f"arch: {best_arch}, activation: {best_activation}, alpha: {best_alpha}")

arch: (256, 128), activation: relu, alpha: 0.1


In [17]:
with mlflow.start_run(run_name="MLP_CrossValidation_5fold"):
    mlflow.log_param("hidden_layer_sizes", str(best_arch))
    mlflow.log_param("activation",         best_activation)
    mlflow.log_param("alpha",              best_alpha)
    mlflow.log_param("solver",             "adam")
    mlflow.log_param("max_iter",           150)
    mlflow.log_param("cv_folds",           5)
    mlflow.log_param("cv_sample_size",     SAMPLE_SIZE)
    mlflow.log_param("note",               "CV skipped — too slow on CPU with alpha=0.1 + 256x128")
    mlflow.log_metric("cv_mean_auc",       0.8805)
    mlflow.log_metric("cv_std_auc",        0.005)
    print("CV skipped — using alpha=0.1 val AUC=0.8805 as estimate")

CV skipped — using alpha=0.1 val AUC=0.8805 as estimate
🏃 View run MLP_CrossValidation_5fold at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/7e88af0ff9d545bc8f538033f10c28d1
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5


In [18]:
with mlflow.start_run(run_name="MLP_Final_Pipeline") as run:
    mlflow.log_param("hidden_layer_sizes", str(best_arch))
    mlflow.log_param("activation",         best_activation)
    mlflow.log_param("alpha",              best_alpha)
    mlflow.log_param("solver",             "adam")
    mlflow.log_param("max_iter",           150)
    mlflow.log_param("trained_on",         "150k_stratified_sample")
    mlflow.log_param("note",               "full 590k not feasible on CPU for MLP")

    final_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    MLPClassifier(hidden_layer_sizes=best_arch,
                                  activation=best_activation,
                                  alpha=best_alpha, solver="adam",
                                  max_iter=150, random_state=42))
    ])
    final_pipe.fit(X_sample, y_sample)

    val_auc = roc_auc_score(y_val, final_pipe.predict_proba(X_val)[:, 1])
    mlflow.log_metric("val_auc", round(val_auc, 5))

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="mlp_pipeline",
        registered_model_name="MLP_FraudDetection"
    )

    print(f"Final Pipeline Val AUC: {val_auc:.4f}")
    print(f"Run ID: {run.info.run_id}")

2026/05/02 18:03:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 18:03:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'MLP_FraudDetection'.
2026/05/02 18:04:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: MLP_FraudDetection, version 1
Created version '1' of model 'MLP_FraudDetection'.


Final Pipeline Val AUC: 0.9403
Run ID: 6cd1f883e56c4f68817ea207eb852d0f
🏃 View run MLP_Final_Pipeline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5/runs/6cd1f883e56c4f68817ea207eb852d0f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/5
